# Southwest Australia — running example for the paper

This notebook is the canonical walk-through we use in the manuscript. It fits the
frequentist `GDM` and the Bayesian `spGDMM` to the Southwest Australia dataset
(94 sites, 865 plant species, Bray–Curtis dissimilarities, 3 environmental
predictors) and generates every figure/table we intend to include for SW.

**Data.** `experiments/white2024_cv/data/southwest_{sites,y}.csv` — the canonical
dataset from the R `gdm` package, pre-computed Bray–Curtis dissimilarities with
`vegan::vegdist`. White et al. (2024) Table 1 reports benchmarks for this exact
split, which we compare against below.

**Model.** Best Bayesian configuration from Table 1:
`spatial_effect="squared_diff"` + `variance="homogeneous"` (Model 7). This matches
White's reported best RMSE/MAE for SW. Priors and sampler settings are frozen by
`CLAUDE.md`: `LogNormal(0, 10)` on `beta`, `Normal(0, 10)` on `beta_sigma`,
`alpha_importance=False`.

> **Warning — do not run MCMC on a login node.** The Bayesian fit cell below is
> gated on `RUN_MCMC = False` and expects a pre-fit NetCDF at
> `results/southwest/southwest_spgdmm_squared_diff_homogeneous.nc`. Produce it via
> `sbatch experiments/white2024_cv/run_bayes_sw.sh` (or any allocation), then
> rerun the notebook.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.model_selection import KFold

from gdmbayes import (
    GDM,
    GDMPreprocessor,
    ModelConfig,
    SamplerConfig,
    crps_boxplot,
    plot_isplines,
    plot_link_curve,
    plot_obs_vs_pred,
    plot_ppc,
    plot_predictor_importance,
    rgb_biological_space,
    spGDMM,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Locate the repo root so paths work whether the notebook is launched from
# examples/ or from the repo root.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "src" / "gdmbayes").exists() else NB_DIR.parent
assert (REPO_ROOT / "src" / "gdmbayes").exists(), f"couldn't find repo root from {NB_DIR}"

DATA_DIR = REPO_ROOT / "experiments" / "white2024_cv" / "data"
RESULTS_DIR = REPO_ROOT / "results" / "southwest"
FIG_DIR = REPO_ROOT / "paper" / "figures" / "southwest"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# Toggle this to True only when running inside a compute allocation with nutpie
# available. See CLAUDE.md: never run MCMC on the login node.
RUN_MCMC = False

print(f"Repo root:   {REPO_ROOT}")
print(f"Data dir:    {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Figure dir:  {FIG_DIR}")

## 2. Load the Southwest Australia data

We use exactly the three predictors White et al. (2024) report for SW:
`phTotal`, `bio5` (max temperature of warmest month), `bio19` (winter
precipitation). Geographic coordinates are `Long`/`Lat` → `xc`/`yc`, with
`time_idx = 0` (single-snapshot dataset).

In [ ]:
ENV_PREDICTORS = ["phTotal", "bio5", "bio19"]

sites = pd.read_csv(DATA_DIR / "southwest_sites.csv")
y = pd.read_csv(DATA_DIR / "southwest_y.csv")["y"].to_numpy()

X = sites.rename(columns={"Long": "xc", "Lat": "yc"}).copy()
X["time_idx"] = 0
X = X[["xc", "yc", "time_idx", *ENV_PREDICTORS]]

n_sites = len(X)
n_pairs = len(y)
print(f"{n_sites} sites · {n_pairs} pairs · y ∈ [{y.min():.3f}, {y.max():.3f}]")
print(f"Saturated pairs (y == 1): {(y == 1.0).sum()}")
X.describe().round(2)

### Site map

Quick look at the spatial layout — 94 sites spread across SW Australia, coloured
by `bio19` (winter precipitation, the strongest gradient in the dataset).

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
sc = ax.scatter(X["xc"], X["yc"], c=X["bio19"], cmap="viridis", s=25, edgecolor="k", lw=0.3)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Southwest Australia — 94 sites")
ax.set_aspect("equal")
fig.colorbar(sc, ax=ax, label="bio19 (winter precip.)", shrink=0.8)
fig.tight_layout()
fig.savefig(FIG_DIR / "sw_site_map.pdf")
plt.show()

## 3. Frequentist GDM (full-data fit)

`GDM` is the sklearn-style frequentist estimator: I-spline features + NNLS.
Settings (`deg=3`, `knots=1`, `geo=True`) match White et al.'s R `gdm` call for
SW so the benchmark comparison is apples-to-apples.

In [ ]:
def make_gdm() -> GDM:
    return GDM(geo=True, splines=3, knots=1)

gdm = make_gdm()
gdm.fit(X, y)
y_pred_gdm = gdm.predict(X)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mae(a, b):
    return float(np.mean(np.abs(a - b)))

corr_gdm, _ = pearsonr(y, y_pred_gdm)
print(f"Deviance explained : {gdm.explained_:.4f}")
print(f"RMSE (train)       : {rmse(y, y_pred_gdm):.4f}")
print(f"MAE  (train)       : {mae(y, y_pred_gdm):.4f}")
print(f"Pearson r (train)  : {corr_gdm:.4f}")
print("\nPredictor importance:")
for k, v in gdm.predictor_importance_.items():
    print(f"  {k:12s} {v:.4f}")

### 10-fold site-level CV (frequentist baseline)

Site-level folds: at each split, test *sites* are held out and every pair
touching a test site is a test pair. `GDMPreprocessor.site_pairs` produces the
pair indices for a given subset of sites.

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)
y_pred_cv = np.full(n_pairs, np.nan)

for fold, (train_sites, test_sites) in enumerate(kf.split(np.arange(n_sites))):
    train_idx = GDMPreprocessor.site_pairs(n_sites, train_sites)
    test_idx = GDMPreprocessor.site_pairs(n_sites, test_sites)
    gdm_cv = make_gdm()
    gdm_cv.fit(X.iloc[train_sites].reset_index(drop=True), y[train_idx])
    y_pred_cv[test_idx] = gdm_cv.predict(X.iloc[test_sites].reset_index(drop=True))

mask = ~np.isnan(y_pred_cv)
freq_cv = {
    "RMSE": rmse(y[mask], y_pred_cv[mask]),
    "MAE": mae(y[mask], y_pred_cv[mask]),
}
print("10-fold CV (frequentist GDM)")
print(f"  RMSE = {freq_cv['RMSE']:.4f}   (White 2024 Ferrier: 0.0737)")
print(f"  MAE  = {freq_cv['MAE']:.4f}   (White 2024 Ferrier: 0.0549)")

## 4. Bayesian spGDMM — full-data fit

Best configuration from White Table 1 for SW:

| spatial_effect | variance    | Model # | RMSE  | CRPS  |
|----------------|-------------|:-------:|:-----:|:-----:|
| squared_diff   | homogeneous |   7     | 0.0731| 0.0414|

The cell below **loads** a pre-fit NetCDF if one exists, and otherwise only
runs MCMC if `RUN_MCMC = True`. To produce the canonical fit, run
`experiments/white2024_cv/run_bayes_sw.sh` under sbatch.

In [ ]:
FIT_TAG = "squared_diff_homogeneous"
fit_path = RESULTS_DIR / f"southwest_spgdmm_{FIT_TAG}.nc"

def build_spgdmm(draws=1000, tune=4000, chains=4) -> spGDMM:
    return spGDMM(
        preprocessor=GDMPreprocessor(
            deg=3, knots=1, mesh_choice="percentile", distance_measure="geodesic"
        ),
        model_config=ModelConfig(
            variance="homogeneous",
            spatial_effect="squared_diff",
            alpha_importance=False,
        ),
        sampler_config=SamplerConfig(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=0.95,
            nuts_sampler="nutpie",
            progressbar=True,
            random_seed=SEED,
        ),
    )

if fit_path.exists():
    print(f"Loading cached fit: {fit_path}")
    model = spGDMM.load(str(fit_path))
elif RUN_MCMC:
    print("No cached fit — running MCMC (only safe inside a compute allocation).")
    model = build_spgdmm()
    model.fit(X, y)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    model.save(str(fit_path))
    print(f"Saved → {fit_path}")
else:
    raise FileNotFoundError(
        f"Expected a cached fit at {fit_path}.\n"
        "Produce it with `sbatch experiments/white2024_cv/run_bayes_sw.sh`,\n"
        "or set RUN_MCMC = True in the setup cell if you're running inside an allocation."
    )

model

### 4a. Convergence and sampling diagnostics

`summarise_sampling()` returns a per-variable table of posterior mean, SD, HDI,
ESS, and R-hat — our first-pass check that NUTS mixed.

In [ ]:
diag = model.summarise_sampling()
rhat_max = float(az.rhat(model.idata_).to_array().max())
print(f"max R-hat across all variables: {rhat_max:.4f}")
diag.head(20)

## 5. Paper figures

### 5a. I-spline effect curves

One figure per predictor: summed I-spline with HDI band. Analogue of
R `gdm::plot.gdm` effect panels. Monotone, saturating to the total predictor
importance at the right endpoint.

In [ ]:
ispline_figs = plot_isplines(model, hdi_prob=0.9)
for (fig, ax) in ispline_figs:
    feature = ax.get_title() or ax.get_xlabel()
    fig.savefig(FIG_DIR / f"sw_ispline_{feature}.pdf")
plt.show()

### 5b. Posterior predictor importance

Bayesian analogue of R `gdm::gdm.varImp`. Bars are posterior means with HDI
intervals; the height equals `sum_j beta[k, j]` (I-splines saturate to 1).

In [ ]:
fig, ax = plot_predictor_importance(model, hdi_prob=0.9, include_distance=True)
fig.savefig(FIG_DIR / "sw_predictor_importance.pdf")
plt.show()

### 5c. Observed vs. predicted and link curve

Two fit diagnostics that mirror R `gdm::plot.gdm` panels 1 and 2:

- **Obs vs. pred** — scatter on the Bray–Curtis scale with the 1:1 line.
- **Link curve** — observed dissimilarity against the linear predictor
  (`eta`) with the model's implied `1 − exp(−eta)` link.

In [ ]:
fig_op, _ = plot_obs_vs_pred(model, X, y)
fig_op.savefig(FIG_DIR / "sw_obs_vs_pred.pdf")

fig_lc, _ = plot_link_curve(model, X, y)
fig_lc.savefig(FIG_DIR / "sw_link_curve.pdf")
plt.show()

### 5d. Posterior predictive check

Overlay of the observed `y` distribution against samples from the posterior
predictive. A well-calibrated model has the observed density within the
posterior predictive envelope.

In [ ]:
fig_ppc, _ = plot_ppc(model, X, y)
fig_ppc.savefig(FIG_DIR / "sw_ppc.pdf")
plt.show()

### 5e. Biological-space RGB map

GDM's community-composition map: transform each site through the fitted
I-splines, project to the top 3 PCs, and rescale to RGB. Sites with similar
predicted community composition get similar colours. We evaluate at the 94
site locations (a continuous climate grid would require interpolating the three
predictors off-site, which is outside this running example).

In [ ]:
X_rgb = X.copy()
X_rgb.index = pd.MultiIndex.from_arrays(
    [X_rgb["yc"].to_numpy(), X_rgb["xc"].to_numpy()], names=["yc", "xc"]
)
rgb = rgb_biological_space(model.idata_, X_rgb, metric="median")

# rgb has dims (time, xc, yc, rgb); pull out per-site colours for a scatter.
rgb_site = rgb.isel(time=0).stack(site=("yc", "xc")).dropna("site", how="all")
colours = np.clip(rgb_site.transpose("site", "rgb").to_numpy(), 0, 1)
coords = np.asarray(rgb_site["site"].to_numpy().tolist())  # (n_site, 2) = (yc, xc)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(coords[:, 1], coords[:, 0], c=colours, s=55, edgecolor="k", lw=0.3)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Biological-space RGB (squared_diff + homogeneous)")
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(FIG_DIR / "sw_biological_space_rgb.pdf")
plt.show()

## 6. CV metrics vs. White et al. (2024) Table 1

If the full Bayesian CV sweep has been run via
`experiments/white2024_cv/run_bayes_sw.sh` (9 configs × 10 folds), the metrics
land at `results/southwest/southwest_cv_metrics.csv`. We load it here and line
it up against the published Table 1 numbers.

In [ ]:
WHITE_TABLE1 = pd.DataFrame(
    [
        ("none", "homogeneous", 0.0439, 0.0790, 0.0595),
        ("none", "covariate_dependent", 0.0435, 0.0805, 0.0608),
        ("abs_diff", "homogeneous", 0.0473, 0.0840, 0.0629),
        ("abs_diff", "covariate_dependent", 0.0454, 0.0820, 0.0626),
        ("squared_diff", "homogeneous", 0.0414, 0.0731, 0.0545),
        ("squared_diff", "covariate_dependent", 0.0407, 0.0748, 0.0556),
    ],
    columns=["spatial_effect", "variance", "CRPS_White", "RMSE_White", "MAE_White"],
)

metrics_path = RESULTS_DIR / "southwest_cv_metrics.csv"
if metrics_path.exists():
    ours = pd.read_csv(metrics_path)
    merged = WHITE_TABLE1.merge(
        ours[["spatial_effect", "variance", "RMSE_CV", "MAE_CV", "CRPS_CV"]],
        on=["spatial_effect", "variance"],
        how="left",
    )
    merged["RMSE_delta"] = merged["RMSE_CV"] - merged["RMSE_White"]
    merged["MAE_delta"] = merged["MAE_CV"] - merged["MAE_White"]
    merged["CRPS_delta"] = merged["CRPS_CV"] - merged["CRPS_White"]
    display(merged.round(4))
    merged.round(4).to_csv(FIG_DIR / "sw_table1_comparison.csv", index=False)
else:
    print(f"(No Bayesian CV metrics yet at {metrics_path} — "
          "run experiments/white2024_cv/run_bayes_sw.sh to populate.)")
    display(WHITE_TABLE1)

## 7. CRPS-by-fold boxplot (optional)

If we also have a per-fold predictions file we can draw the CRPS boxplot used
in the manuscript. Otherwise skip — this cell is a no-op.

In [ ]:
fold_csv = RESULTS_DIR / "southwest_cv_fold_metrics.csv"
if fold_csv.exists():
    fold_df = pd.read_csv(fold_csv)
    fig, ax = crps_boxplot(fold_df)
    fig.savefig(FIG_DIR / "sw_crps_boxplot.pdf")
    plt.show()
else:
    print(f"(No per-fold file at {fold_csv} — skipping CRPS boxplot.)")

## 8. Figure inventory

Everything the notebook dropped into `paper/figures/southwest/` — this is the
set of files to reference from `paper/manuscript.tex`.

In [ ]:
for p in sorted(FIG_DIR.iterdir()):
    print(f"  {p.relative_to(REPO_ROOT)}")